In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
import os
import datetime

In [ ]:
# 📌 Set Chrome's download directory
download_dir = os.path.expanduser("~/Desktop/efas/tefas")  # Change if needed
options = webdriver.ChromeOptions()
prefs = {"download.default_directory": download_dir}  # Set Chrome download path
options.add_experimental_option("prefs", prefs)

# 📌 Start WebDriver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


try:
    # Open TEFAS website
    driver.get("https://www.tefas.gov.tr/TarihselVeriler.aspx")

    # Select "Menkul Kıymet Yatırım Fonları"
    time.sleep(2)
    fon_tipi_select = wait.until(EC.element_to_be_clickable((By.ID, "RadioButtonListFundMainType_0")))
    fon_tipi_select.click()

    # Define the date range
    start_date = datetime.datetime.strptime("2025-02-28", "%Y-%m-%d")
    end_date = datetime.datetime.strptime("2025-03-09", "%Y-%m-%d")
    current_date = start_date

    while current_date <= end_date:
        formatted_date = current_date.strftime("%d.%m.%Y")
        file_date = current_date.strftime("%Y%m%d")
        print(f"\n🔍 Processing date: {formatted_date}")

        # Enter Start and End Dates
        tarih_input = wait.until(EC.presence_of_element_located((By.ID, "TextBoxStartDate")))
        tarih_input.clear()
        tarih_input.send_keys(formatted_date)

        tarih_input = wait.until(EC.presence_of_element_located((By.ID, "TextBoxEndDate")))
        tarih_input.clear()
        tarih_input.send_keys(formatted_date)

        # Click "Ara" button
        ara_button = wait.until(EC.element_to_be_clickable((By.ID, "ButtonSearchDates")))
        ara_button.click()

        # Wait for results to load
        time.sleep(10)

        # Check if table says "Kayıt yok"
        table_info = driver.find_element(By.ID, "table_general_info_info").text
        if "Kayıt yok" in table_info:
            print(f"⚠️ No data available for {formatted_date}. Skipping...")
            current_date += datetime.timedelta(days=1)
            continue

        # Download Fund Returns CSV (Genel Bilgiler)
        try:
            csv_button = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(@class, 'buttons-csv')]")))
            driver.execute_script("arguments[0].click();", csv_button)
            print("🎉 CSV Download Started for Fund Returns!")
            time.sleep(5)

            # Rename downloaded file
            files = os.listdir(download_dir)
            csv_files = [f for f in files if f.endswith(".csv")]

            if csv_files:
                latest_file = max(csv_files, key=lambda f: os.path.getctime(os.path.join(download_dir, f)))
                old_path = os.path.join(download_dir, latest_file)

                # Rename File
                new_filename = f"{file_date}_tefas_fon.csv"
                new_path = os.path.join(download_dir, new_filename)
                os.rename(old_path, new_path)
                print(f"✅ File renamed to: {new_filename}")
        except Exception as e:
            print(f"⚠️ Could not download Fund Returns CSV for {formatted_date}. Error: {e}")

        # Switch to "Portföy Dağılımı" tab
        portfolio_tab = wait.until(EC.element_to_be_clickable((By.ID, "ui-id-2")))
        portfolio_tab.click()
        print("📌 Switched to 'Portföy Dağılımı' tab!")

        # Wait for the table to load
        try:
            wait.until(EC.visibility_of_element_located((By.ID, "table_allocation")))
            print("📊 'Portföy Dağılımı' table is visible!")
        except:
            print("⚠️ Could not detect 'Portföy Dağılımı' table. Trying to continue anyway...")

        time.sleep(3)  # Additional wait to ensure elements are fully loaded

        # Download Fund Allocations CSV (Portföy Dağılımı)
        try:
            # Wait for the CSV button to be present and clickable
            csv_button = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(@class, 'buttons-csv')]")))
            driver.execute_script("arguments[0].click();", csv_button)
            print("🎉 CSV Download Started for Fund Allocations!")
            time.sleep(5)

            # Rename downloaded file
            files = os.listdir(download_dir)
            csv_files = [f for f in files if f.endswith(".csv")]

            if csv_files:
                latest_file = max(csv_files, key=lambda f: os.path.getctime(os.path.join(download_dir, f)))
                old_path = os.path.join(download_dir, latest_file)

                # Rename File
                new_filename = f"{file_date}_tefas_dag.csv"
                new_path = os.path.join(download_dir, new_filename)
                os.rename(old_path, new_path)
                print(f"✅ File renamed to: {new_filename}")
        except Exception as e:
            print(f"⚠️ Could not download Fund Allocations CSV for {formatted_date}. Error: {e}")

        # Switch back to "Genel Bilgiler" tab before moving to the next date
        try:
            genel_tab = wait.until(EC.element_to_be_clickable((By.ID, "ui-id-1")))
            genel_tab.click()
            print("🔄 Switched back to 'Genel Bilgiler' tab!")
            time.sleep(2)
        except:
            print("⚠️ Could not switch back to 'Genel Bilgiler' tab!")

        # Move to the next date
        current_date += datetime.timedelta(days=1)

except Exception as e:
    print(f"❌ Error: {e}")

finally:
    time.sleep(5)
    driver.quit()